In [1]:
cd ./wrkDir/s19/s19-1

/work/xiangchong.li/work/galSim-HSC/s19/s19-1


In [14]:
%pylab inline
import os
import gc
import sys
import numpy as np
import astropy.io.ascii as ascii
import pandas as pd
import scipy.stats as stats
import catutil as utilities
from scipy.interpolate import griddata

import matplotlib
import matplotlib.pyplot as plt

anaDir='./anaCat_newS19Mask_fdeltacut/'
assert os.path.isdir(anaDir)
pltDir=os.path.join(anaDir,'plot/sanityTest_withZ/')
if not os.path.isdir(pltDir):
    os.mkdir(pltDir)
inFname=os.path.join(anaDir,'allSim.h5')
assert os.path.isfile(inFname)

Populating the interactive namespace from numpy and matplotlib


In [4]:
dataP =   pd.read_hdf(inFname,mode='r')
data=dataP.to_records()

min_snr=10
max_snr=None
max_magA10=25.5
min_res=0.3
max_mag=24.5
max_logb=-0.38
max_sigma_e=0.4
max_e=2.

wlmask2    =   utilities.galaxy_selector(catalog=data,
                min_snr=min_snr, #snr
                max_snr=max_snr,
                max_magA10=max_magA10, #magA10
                min_res=min_res, #res
                max_mag=max_mag, #mag
                max_logb=max_logb,#logb
                force_90_pair=True,
                max_e=max_e,     #absE
                max_sigma_e=max_sigma_e) #sigma_e

dataP=dataP[wlmask2&(~np.isnan(dataP['mizuki_Z']))]
data=dataP.to_records()

In [5]:
indexS=  dataP.index.levels[1]
indexList= ascii.read('info_newS19Mask_fd001_sim.csv')['num']
mask=np.in1d(indexList,indexS)
indexList=indexList[mask]

In [7]:
outData=[]
for index in indexList:
    catalog =   dataP.xs(index,level='isim').to_records(index=False)
    reweight     =   catalog['weight'][0]
    if len(catalog)<50 or reweight<1.e-6:
        continue
    # Force the same weight and Response
    e_rms = utilities.get_erms_model(catalog)
    shape_weights = utilities.get_weight_model(catalog)*catalog['weight']
    
    # True shear
    g1_true, g2_true= utilities.get_true_shear(catalog[0])
    # PSF shapes
    psf_e1, psf_e2  =   utilities.get_psf_ellip(catalog[0])
    # Shear Estimation
    e1,e2   =   utilities.get_gal_ellip(catalog)
    shear_num_1 = np.sum(shape_weights * e1)
    shear_num_2 = np.sum(shape_weights * e2)
    sum_weight = np.sum(shape_weights)
    # Calibration
    model_m = utilities.get_m_model(catalog,weight_bias=True,photo_z_dep=True)

    model_c1,model_c2 = utilities.get_c_model(catalog,weight_bias=True,photo_z_dep=True)
    bias_num = np.sum(shape_weights*model_m)
    add_num_1 = np.sum(shape_weights*model_c1)
    add_num_2 = np.sum(shape_weights*model_c2)
    bias = bias_num / sum_weight
    resp_num = np.sum(shape_weights * e_rms**2)
    responsivity = 1.0 - resp_num/sum_weight

    g1 = shear_num_1 / (2.0*responsivity*sum_weight*(1.0+bias)) \
        - add_num_1/((1.0+bias)*sum_weight)
    g2 = shear_num_2 / (2.0*responsivity*sum_weight*(1.0+bias)) \
        - add_num_2/((1.0+bias)*sum_weight)
    sigma= 0.25/np.sqrt(len(catalog))
    
    oDat    =   (g1_true,g2_true,g1,g2,sigma,psf_e1,psf_e2,reweight)
    outData.append(oDat)
    
odf=pd.DataFrame(outData,columns=('g1_true','g2_true','g1','g2','sigma','psf_e1','psf_e2','reweight')).to_records()
division_val = np.sqrt(odf['reweight'])
division_val[division_val<1.e-6] = 1.e-6
odf['sigma'] /= division_val
# remove unrealistic sim
odf=odf[(odf['sigma'] < 1.e+10)]
outfname=os.path.join(pltDir,'shear_est_subfields.csv')
ascii.write(odf,outfname,overwrite=True)

In [26]:
odf=ascii.read(outfname)
# get multiplicative bias and additive bias
m1,a1,sigm1,siga1= utilities.get_m_c_a(
    odf['g1'],odf['g2'],
    odf['g1_true'],odf['g2_true'],
    odf['psf_e1'],odf['psf_e2'],
    odf['sigma'],separate_components=False)
print('m = %.4f +/- %.4f' % (m1, sigm1))
print('a = %.4f +/- %.4f' % (a1, siga1))

m = -0.0006 +/- 0.0016
a = 0.0000 +/- 0.0006


In [17]:
outfname=os.path.join(pltDir,'shear_est_subfields.csv')
odf=ascii.read(outfname)
nn=len(odf)
nran=200
m1All=np.empty(nran)
a1All=np.empty(nran)
for i in range(nran):
    np.random.seed(i)
    inds=np.random.randint(0,nn,nn)
    odfTmp=odf[inds]
    m1,a1,sigm1,siga1= utilities.get_m_c_a(
        odfTmp['g1'],odfTmp['g2'],
        odfTmp['g1_true'],odfTmp['g2_true'],
        odfTmp['psf_e1'],odfTmp['psf_e2'],
        odfTmp['sigma'],separate_components=False)
    m1All[i]=m1
    a1All[i]=a1
    del inds,odfTmp,m1,a1
    gc.collect()

In [18]:
np.average(m1All),np.std(m1All),np.std(a1All)

(-0.00041021704276507267, 0.001845331132381345, 0.0007353819989694262)

In [37]:
import scipy
def systematics_func(x, m, c):
    return m * x + c
g = np.concatenate([odf['g1'], odf['g2']])
g_true = np.concatenate([odf['g1_true'], odf['g2_true']])
sigma_vals = np.concatenate([odf['sigma'], odf['sigma']])

popt1, pcov1 = scipy.optimize.curve_fit(systematics_func,
                        g_true,
                        g-g_true,sigma=sigma_vals)

In [35]:
odf['sigma'].min()

0.0013989473493308886

In [38]:
popt1

array([-4.38729863e-04,  7.98079795e-05])

In [36]:
pcov1

array([[5.14296331e-06, 3.75814377e-09],
       [3.75814377e-09, 1.57206113e-09]])